# 05 · Extra — Conteo de pasajeros visibles
**IAM-811** · Laboratorio Final de Hito

Se usa **segmentación de instancias** YOLO11-seg (clase `person` de COCO) y se cuentan las personas cuya caja
está mayoritariamente dentro de la caja del vehículo principal (≥ 60 %; en moto ≥ 30 % porque el conductor sobresale).

**Privacidad:** no hay reconocimiento facial, no se guardan recortes de rostros; solo se devuelve el número
(`pasajeros`) y las cajas para dibujar. La web guarda únicamente `pasos.pasajeros`.

Este notebook permite ajustar los umbrales con fotos propias y medir el error de conteo.

In [ ]:
!pip -q install -U ultralytics==8.4.218
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'

In [ ]:
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from ultralytics import YOLO, YOLOE
personas = YOLO('yolo11s-seg.pt')
vehiculos = YOLOE('yoloe-11l-seg.pt')
PROMPTS = ['car', 'van', 'minibus', 'bus', 'truck', 'motorcycle']
vehiculos.set_classes(PROMPTS, vehiculos.get_text_pe(PROMPTS))
CONF_PERSONA, UMBRAL_DENTRO, UMBRAL_DENTRO_MOTO = 0.35, 0.6, 0.3

def fraccion_dentro(a, b):
    x1, y1, x2, y2 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    return max(0, x2 - x1) * max(0, y2 - y1) / max(1e-6, (a[2] - a[0]) * (a[3] - a[1]))

def contar(img):
    rv = vehiculos.predict(img, conf=0.3, verbose=False)[0]
    if rv.boxes is None or len(rv.boxes) == 0:
        return 0, None, []
    cajas = rv.boxes.xyxy.cpu().numpy()
    i = int(np.argmax((cajas[:, 2] - cajas[:, 0]) * (cajas[:, 3] - cajas[:, 1])))
    veh, clase = cajas[i].tolist(), rv.names[int(rv.boxes.cls[i])]
    umbral = UMBRAL_DENTRO_MOTO if clase == 'motorcycle' else UMBRAL_DENTRO
    rp = personas.predict(img, conf=CONF_PERSONA, classes=[0], verbose=False)[0]
    dentro = [b.tolist() for b in rp.boxes.xyxy.cpu().numpy() if fraccion_dentro(b, veh) >= umbral]
    return len(dentro), veh, dentro

In [ ]:
from google.colab import files
subidos = files.upload()     # fotos + (opcional) pasajeros_reales.csv con columnas archivo,pasajeros
for n in subidos:
    if n.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = cv2.imread(n)
        k, veh, ps = contar(img)
        if veh:
            cv2.rectangle(img, tuple(map(int, veh[:2])), tuple(map(int, veh[2:])), (0, 255, 0), 3)
        for b in ps:
            cv2.rectangle(img, tuple(map(int, b[:2])), tuple(map(int, b[2:])), (255, 255, 0), 2)
        plt.figure(figsize=(8, 5)); plt.imshow(img[:, :, ::-1]); plt.title(f'{n}: {k} pasajeros'); plt.axis('off'); plt.show()

In [ ]:
if os.path.exists('pasajeros_reales.csv'):
    gt = pd.read_csv('pasajeros_reales.csv')
    gt['predicho'] = [contar(cv2.imread(a))[0] for a in gt['archivo']]
    print('MAE:', (gt['predicho'] - gt['pasajeros']).abs().mean(), '| exactos:', (gt['predicho'] == gt['pasajeros']).mean())
    display(gt)